# ANEEL RAG — Módulo 5: Embedding e Indexação

Rode as células **em ordem**, uma por vez.

**Pré-requisito:** o arquivo `chunks.parquet` já deve estar no seu Google Drive em:
`Meu Drive/ANEEL-RAG/chunks.parquet`

In [ ]:
# CÉLULA 1 — Verifica a GPU disponível
# Deve mostrar Tesla T4 (ou melhor). Se mostrar 'No GPU', vá em:
# Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4
!nvidia-smi

In [ ]:
# CÉLULA 2 — Monta o Google Drive
# Uma janela vai pedir permissão — clique em 'Conectar ao Google Drive'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CÉLULA 3 — Clona o repositório do GitHub
# Substitua pela URL do repositório do grupo se for diferente
REPO_URL = "https://github.com/murilohonorato/ANEEL-RAG.git"

!git clone {REPO_URL} /content/ANEEL-RAG
%cd /content/ANEEL-RAG
!git log --oneline -3

In [ ]:
# CÉLULA 4 — Copia o chunks.parquet do Drive para o projeto
import os

src  = "/content/drive/MyDrive/ANEEL-RAG/chunks.parquet"
dest = "/content/ANEEL-RAG/data/processed/chunks.parquet"

os.makedirs("/content/ANEEL-RAG/data/processed", exist_ok=True)

if not os.path.exists(src):
    raise FileNotFoundError(
        f"Arquivo não encontrado: {src}\n"
        "Verifique se o chunks.parquet está em 'Meu Drive/ANEEL-RAG/chunks.parquet'"
    )

!cp "{src}" "{dest}"
size_mb = os.path.getsize(dest) / 1e6
print(f"✓ chunks.parquet copiado ({size_mb:.0f} MB)")

In [ ]:
# CÉLULA 5 — (OPCIONAL) Copia qdrant_db do Drive para retomar progresso anterior
# Só rode esta célula se você já tem um qdrant_db.zip no Drive de uma sessão anterior.
# Se for a primeira vez, PULE esta célula.

import os
zip_path = "/content/drive/MyDrive/ANEEL-RAG/qdrant_db.zip"

if os.path.exists(zip_path):
    !unzip -q "{zip_path}" -d /content/ANEEL-RAG/
    print("✓ qdrant_db restaurado — modo incremental ativo")
else:
    print("Nenhum qdrant_db.zip encontrado — começando do zero")

In [ ]:
# CÉLULA 6 — Instala as dependências necessárias
# Torch com CUDA já vem no Colab — só instalamos o restante
!pip install -q FlagEmbedding==1.4.0 qdrant-client==1.17.1 loguru python-dotenv tqdm
print("✓ Dependências instaladas")

In [ ]:
# CÉLULA 7 — Verifica torch com CUDA e o modelo
import torch
print(f"Torch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")

In [ ]:
# CÉLULA 8 — Roda o embedding e indexação
#
# Flags usadas:
#   --batch-size 128  → T4 tem 16 GB de VRAM, comporta batches grandes
#   --reset           → recria o índice do zero (remova se retomando com qdrant_db.zip)
#
# Se estiver retomando de sessão anterior (célula 5 rodou com sucesso),
# troque --reset por --incremental

import os
os.chdir("/content/ANEEL-RAG")

!python src/05_embed_index.py --reset --batch-size 128

In [ ]:
# CÉLULA 9 — Salva o qdrant_db no Google Drive
# Rode esta célula assim que a célula 8 terminar.
# Gera um qdrant_db.zip no seu Drive para você baixar depois.

import os
output_zip = "/content/drive/MyDrive/ANEEL-RAG/qdrant_db.zip"

print("Comprimindo qdrant_db/ ...")
!cd /content/ANEEL-RAG && zip -r "{output_zip}" qdrant_db/

size_mb = os.path.getsize(output_zip) / 1e6
print(f"✓ Salvo em {output_zip} ({size_mb:.0f} MB)")
print("Agora baixe o arquivo do Google Drive para seu PC.")

In [ ]:
# CÉLULA 10 — (OPCIONAL) Salva também o index_stats.json
!cp /content/ANEEL-RAG/data/processed/index_stats.json \
   "/content/drive/MyDrive/ANEEL-RAG/index_stats.json"
print("✓ index_stats.json salvo no Drive")